In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}.football_data_org_competitions"
)

SILVER_TABLE = (
    f"{SILVER_SCHEMA}.football_data_org_competitions"
)

# One Silver row represents one competition.
BUSINESS_KEY = [
    "competition_id",
]

REQUIRED_SILVER_COLUMNS = [
    "competition_id",
    "competition_code",
    "competition_name",
    "league_code",
]


# -----------------------------------------------------------------------------
# Read Bronze
# -----------------------------------------------------------------------------

# Bronze contains append-only snapshots of competition API responses.
# Missing Bronze data should fail rather than silently produce an empty Silver.

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

bronze_df = spark.table(BRONZE_TABLE)


# -----------------------------------------------------------------------------
# Validate required Bronze columns
# -----------------------------------------------------------------------------

REQUIRED_BRONZE_COLUMNS = [
    "id",
    "code",
    "name",
    "type",
    "emblem",
    "area",
    "currentSeason",
    "lastUpdated",
    "_league",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)

# Validate nested fields explicitly because top-level validation
# does not inspect columns inside structs.

REQUIRED_NESTED_COLUMNS = [
    "area.id",
    "area.name",
    "currentSeason.id",
    "currentSeason.startDate",
    "currentSeason.endDate",
    "currentSeason.currentMatchday",
]

for column_path in REQUIRED_NESTED_COLUMNS:
    try:
        bronze_df.select(F.col(column_path))
    except Exception as exc:
        raise ValueError(
            f"Required nested Bronze column is missing: {column_path}"
        ) from exc


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------

# Nested strings are cleaned individually during Silver selection.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate source identity
# -----------------------------------------------------------------------------

unexpected_source_rows = (
    bronze_df
    .filter(
        F.col("_source_system").isNull()
        | (F.col("_source_system") != F.lit(SOURCE_SYSTEM))
    )
    .limit(1)
    .count()
)

if unexpected_source_rows > 0:
    raise ValueError(
        f"Unexpected source system found in {BRONZE_TABLE}."
    )


# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------

silver_df = (
    bronze_df
    .select(

        # Competition identifiers and attributes

        F.col("id")
        .cast("int")
        .alias("competition_id"),

        clean_string("code")
        .alias("competition_code"),

        clean_string("name")
        .alias("competition_name"),

        clean_string("type")
        .alias("competition_type"),

        clean_string("emblem")
        .alias("competition_emblem"),


        # Geographic information

        F.col("area.id")
        .cast("int")
        .alias("area_id"),

        clean_string("area.name")
        .alias("area_name"),


        # Current competition season

        F.col("currentSeason.id")
        .cast("int")
        .alias("current_season_id"),

        F.col("currentSeason.startDate")
        .cast("date")
        .alias("current_season_start_date"),

        F.col("currentSeason.endDate")
        .cast("date")
        .alias("current_season_end_date"),

        F.col("currentSeason.currentMatchday")
        .cast("int")
        .alias("current_matchday"),


        # Additional competition attributes

        (
            F.col("numberOfAvailableSeasons")
            if "numberOfAvailableSeasons" in bronze_df.columns
            else F.lit(None)
        )
        .cast("int")
        .alias("number_of_available_seasons"),

        F.col("lastUpdated")
        .cast("timestamp")
        .alias("last_updated"),


        # Canonical platform league identifier

        clean_string("_league")
        .alias("league_code"),


        # Bronze lineage

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Validate competition-to-league mapping
# -----------------------------------------------------------------------------

# The API competition code must match the canonical platform mapping.
# This catches incorrect Bronze league assignments and unexpected codes.

expected_league = map_values(
    "competition_code",
    FOOTBALL_DATA_ORG_COMPETITIONS,
)

invalid_mapping_rows = (
    silver_df
    .filter(
        expected_league.isNull()
        | F.col("league_code").isNull()
        | (F.col("league_code") != expected_league)
    )
    .limit(1)
    .count()
)

if invalid_mapping_rows > 0:
    raise ValueError(
        "Competition code and canonical league mapping are inconsistent."
    )


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------

# Keep the latest ingested version of each competition.
# Bronze remains append-only; Silver represents the latest known state.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Validate competition-code uniqueness
# -----------------------------------------------------------------------------

# The API competition ID is the business key, but competition codes should
# also be unique across our selected competitions.

duplicate_codes = (
    silver_df
    .groupBy("competition_code")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_codes > 0:
    raise ValueError(
        "Duplicate competition codes found after deduplication."
    )


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)

if validation_metrics["row_count"] == 0:
    raise ValueError(
        "Silver competitions dataset is empty. "
        "Refusing to overwrite the target table."
    )


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------

# Full refresh is appropriate for the current project phase.
# Repeated executions replace the target state instead of appending rows.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"competition records into {SILVER_TABLE}."
)
